# Reasoning and Efficient Model Inference

Compare reasoning budgets, verifier-backed search, and speculative decoding under explicit quality and latency constraints.

Run the lecture cells in sequence with Python 3.10 or later. The default exercises use bundled data and require no model account.


## Practice case · Fast answers without a guess

Meera owns HarborLink's response-time budget. Some requests only need one current policy excerpt. Others require choosing a route across stations with exclusions. She can afford added model time when it changes what the desk can verify.

The workplace scenario is fictional.


## Allocate compute at inference time


### Lecture 01 · Reasoning budgets and model cascades

Design a request-level escalation policy and compare it against always using one model configuration.

#### Pick an observable escalation signal

A cheap candidate returns a confident answer about a closed station. Its source date is older than the date supplied in the request.

Use verifiable source age as an escalation signal. Benchmark the whole escalation path, including both attempts and the capacity needed for a manual handoff.

#### How it works

Some hosted models expose a reasoning-effort setting. A service can also route between smaller and larger models or vary a search budget. Extra compute has diminishing returns and can increase tail latency. Use a pre-specified escalation signal, such as failure to produce a verifiable answer or a difficult input slice. An internally reported confidence score needs calibration before it can drive a cascade.

##### Spend the budget where you can measure it
Choose a quality gate that can be checked against the request and its authoritative evidence. Pin the prompt, endpoint version, and settings for the first and second attempts. Report verified successes divided by eligible requests and spend divided by verified successes. A cheaper average request can still produce a more expensive usable result. Use a fallback path when the remaining latency budget cannot fit another attempt.

An inference budget chooses how much compute to use for a deployed checkpoint. Post-training methods change the checkpoint itself. For example, reinforcement learning from verifiable rewards can use an outcome checker to score training responses. If you compare checkpoints as well as budgets, change one axis at a time. Include checks for whether the outputs satisfy the intended task beyond the reward signal.

#### Worked example

A transit analyst uses a cheap extraction pass for routine tickets. Tickets with conflicting documents reach a more expensive evidence check. If both fail the answer contract, the workflow hands off.

#### Try it

Compare always-cheap, always-expensive, and cascade strategies on the same workload. Count both attempts for escalated requests.


In [ ]:
# These fixtures describe verification results, not sampled model responses.
requests = [(True, True), (True, True), (False, True), (False, False)]


def trial(rows, mode):
    verified = spend = escalations = 0
    for cheap, expensive in rows:
        if mode == "expensive":
            spend += 4
            verified += expensive
            continue
        spend += 1
        if mode == "cascade" and not cheap:
            escalations += 1
            spend += 4
            verified += expensive
        else:
            verified += cheap
    return verified, spend, escalations


for mode in ("cheap", "expensive", "cascade"):
    print(mode, trial(requests, mode))
assert trial(requests, "cascade") == (3, 12, 2)


**Compare your result**

Escalation cost is the sum of both attempts. Report hard-slice quality and the handoff rate alongside the overall score.


### Lecture 02 · Bounded search and independent verification

Select a valid answer from candidate outputs under a bounded attempt budget.

#### Test the answer before reading the explanation

Two routes look reasonable in the planner's prose. One uses a closed lift and the other has a bad total.

Run an independent feasibility check over the structured routes. Track when a valid route is never proposed separately from when the verifier accepts a bad route.

#### How it works

A generator proposes candidates, and a separate verifier checks observable conditions. For deterministic tasks, a test or solver can verify an answer. A learned judge adds another fallible estimate. Increasing sample count helps only when candidates add useful diversity and the verifier recognizes valid answers.

##### A verifier can fail too
With a deterministic verifier, make the acceptance predicate part of the test suite. Use deliberately flawed candidates, including one whose stated answer is right while its action is infeasible. If a judge must read free text, calibrate it against human judgments and test whether irrelevant formatting or wording affects its labels. Track generator coverage, false acceptances, and out-of-budget cases separately.

#### Worked example

A route planner proposes traces. The verifier checks which edges exist, whether any station is closed, and whether the reported total matches the route. A polished rationale cannot substitute for these checks.

#### Try it

Add a plausible but infeasible candidate before a valid one. Compare the selection at budgets one and three.


In [ ]:
edges = {("West", "Lift"): 2, ("Lift", "East"): 2, ("West", "East"): 7}
proposals = [(("West", "Lift", "East"), 4), (("West", "East"), 6), (("West", "East"), 7)]


def choose_route(candidates, budget, closed):
    for nodes, total in candidates[:budget]:
        segments = list(zip(nodes, nodes[1:]))
        if len(nodes) < 2 or (nodes[0], nodes[-1]) != ("West", "East") or closed.intersection(nodes):
            continue
        if all(segment in edges for segment in segments) and sum(edges[edge] for edge in segments) == total:
            return nodes, total
    return None


assert choose_route(proposals, 1, {"Lift"}) is None
assert choose_route(proposals, 3, {"Lift"}) == (("West", "East"), 7)
print(choose_route(proposals, 3, {"Lift"}))


**Compare your result**

A budget of one may abstain. A larger budget can find a verified candidate. A feasibility check alone does not prove optimality.


## Improve throughput and measure honestly


### Lecture 03 · Prefill, decode and speculative decoding

Model draft-token acceptance and explain when speculation can lower decode time.

#### Reject the suffix when the prefix changes

The draft system expects a policy sentence to end with 'business days'. The target chooses a token that begins a different phrase.

Keep the matching prefix and continue from the target's chosen token. Any later draft token was proposed in an obsolete context.

#### How it works

Prefill processes the prompt. Decode produces incremental continuations and accesses cached state. A speculative system proposes several draft tokens and asks the target to check them together. In greedy decoding, accept matching tokens through the first mismatch and emit the target token at that position. Exact stochastic sampling requires acceptance and correction using the draft and target distributions.

##### Keep comparison conditions fixed
Draft and target can have different compute costs and batching behavior. Log how many proposed tokens are accepted per check and compare against a non-speculative baseline on the same output lengths. Greedy verification is a useful teaching limit. For sampling, equality to one sampled target continuation is not the acceptance rule, and dropping the correction step can change the output distribution.

#### Worked example

A four-token draft with a mismatch at position three commits the first two draft tokens and the target's third token. The last draft token is discarded because it depends on a different prefix.

#### Try it

Compare identical and mismatched drafts with the same greedy target fixtures. Explain why committed tokens per check is only a proxy for speed.


In [ ]:
def greedy_speculation(draft, target):
    """One check. Target must supply a token for each position verified."""
    if len(target) < len(draft):
        raise ValueError("Missing target tokens")
    for index, token in enumerate(draft):
        if token != target[index]:
            return tuple(draft[:index]) + (target[index],), index
    return tuple(draft), len(draft)


assert greedy_speculation(("a", "b", "x", "d"), ("a", "b", "c", "z")) == (("a", "b", "c"), 2)
assert greedy_speculation(("a", "b"), ("a", "b")) == (("a", "b"), 2)
print(greedy_speculation(("a", "b", "x", "d"), ("a", "b", "c", "z")))


**Compare your result**

Draft work, verifier cost, batching and workload shape can overturn the speedup. Preserve the target's decoding contract when measuring latency.


### Lecture 04 · Serving workload and release gates

Report time to first token, inter-token latency, and verified quality under a predeclared serving gate.

#### A warm benchmark misses the expensive cohort

A performance report looks good on repeated policy prompts. A contractor batch uses fresh logs, increasing the queue time for first tokens.

Fix separate gates for warm and cold cohorts. Record prompt lengths and concurrency so a follow-up can reproduce the failure.

#### How it works

Continuous batching can admit a new request between generation steps. Prefix caching can reuse prefill work for a compatible model and prefix. KV paging manages allocation, while quantization changes storage or arithmetic. Prefill/decode disaggregation adds a transfer path. These mechanisms target different bottlenecks. Benchmark warm and cold requests, concurrency, input length and output length under the same quality contract.

##### Report tails with their workloads
Keep warm cache, cold cache, short prompt, long prompt, low concurrency, and saturation visible in the benchmark. Time to first token covers queueing and prefill. Inter-token timing captures cadence during decode. Record total end-to-end latency as its own measure. The output from the faster route must still pass the same frozen functional checks. Bottlenecks can move when a batching or caching change makes another stage busier.

#### Worked example

A cache improves repeated short prompts. A cohort with long unique prompts gains little and may experience worse p95 time to first token because it competes for capacity.

#### Try it

Apply one release gate to the candidate fixtures. Inspect what happens when the warm and cold cohorts are pooled.


In [ ]:
from math import ceil


def p95(observations):
    return sorted(observations)[ceil(0.95 * len(observations)) - 1]


# Synthetic milliseconds. Pooling makes the smaller cohort's failure disappear.
warm = [85] * 100
cold = [140, 210, 310]
limit = 200
assert p95(warm + cold) <= limit < p95(cold)
print("p95 TTFT by cohort", {"warm": p95(warm), "cold": p95(cold), "pooled": p95(warm + cold)})


**Compare your result**

Gate each critical cohort explicitly. Avoid describing fixture timing as measured latency or conflating prefill latency with total time.


## Course project

Deliver a paired inference evaluation with a default, an escalation policy, and a bounded fallback.

Report quality, p95 latency, cost per verified success, and the escalation rate separately. Keep model and serving configurations versioned.
